# Data Profiling — Workshop-1: Recruitment Dimensional Data Warehouse


In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from extract import extract

pd.set_option("display.max_columns", None)

df = extract()
df.shape

2026-09-03 23:45:49,600 [INFO] Extracting raw data from C:\Users\Daniel\Desktop\UAO\V Semestre\ETL\ETL_2026-2_Workshop-1\data\raw\candidates.csv
2026-09-03 23:45:49,673 [INFO] Extracted 50000 rows and 10 columns


(50000, 10)

## 1. Rows, columns and column names

In [2]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print()
print("Column names:")
for c in df.columns:
    print(f"  - {c}")

Rows: 50,000
Columns: 10

Column names:
  - First Name
  - Last Name
  - Email
  - Application Date
  - Country
  - YOE
  - Seniority
  - Technology
  - Code Challenge Score
  - Technical Interview Score


## 2. Data types

In [3]:
df.dtypes

First Name                   string
Last Name                    string
Email                        string
Application Date                str
Country                      string
YOE                           int64
Seniority                    string
Technology                   string
Code Challenge Score          int64
Technical Interview Score     int64
dtype: object

## 3. Missing values

In [4]:
missing = df.isnull().sum()
missing[missing > 0] if missing.sum() > 0 else print("No missing values in any column.")

No missing values in any column.


## 4. Duplicate records

In [5]:
print("Exact duplicate rows (all columns identical):", df.duplicated().sum())
print("Rows sharing the same Email (repeat applicants):", df.duplicated(subset=['Email']).sum())

Exact duplicate rows (all columns identical): 0
Rows sharing the same Email (repeat applicants): 167


**Finding:** there are no fully duplicated rows, but 167 rows share an email
with another row — these are candidates who applied more than once (different
`Application Date` / `Technology` / scores). Since the declared grain of the
fact table is **one row per application**, these are kept as separate,
legitimate facts, not deduplicated.

## 5. Unique values in relevant categorical attributes

In [6]:
print("Seniority levels:", sorted(df['Seniority'].unique()))
print()
print("Number of unique Technology values:", df['Technology'].nunique())
print(sorted(df['Technology'].unique()))
print()
print("Number of unique Country values:", df['Country'].nunique())

Seniority levels: ['Architect', 'Intern', 'Junior', 'Lead', 'Mid-Level', 'Senior', 'Trainee']

Number of unique Technology values: 24
['Adobe Experience Manager', 'Business Analytics / Project Management', 'Business Intelligence', 'Client Success', 'Data Engineer', 'Database Administration', 'Design', 'DevOps', 'Development - Backend', 'Development - CMS Backend', 'Development - CMS Frontend', 'Development - Frontend', 'Development - FullStack', 'Game Development', 'Mulesoft', 'QA Automation', 'QA Manual', 'Sales', 'Salesforce', 'Security', 'Security Compliance', 'Social Media Community Management', 'System Administration', 'Technical Writing']

Number of unique Country values: 244


## 6. Application Date range

In [7]:
dates = pd.to_datetime(df['Application Date'], errors='coerce')
print("Min date:", dates.min().date())
print("Max date:", dates.max().date())
print("Unparseable dates:", dates.isna().sum())

Min date: 2018-01-01
Max date: 2022-07-04
Unparseable dates: 0


## 7. Score ranges (Code Challenge / Technical Interview)

In [8]:
print("Code Challenge Score: min =", df['Code Challenge Score'].min(), "max =", df['Code Challenge Score'].max())
print("Technical Interview Score: min =", df['Technical Interview Score'].min(), "max =", df['Technical Interview Score'].max())

Code Challenge Score: min = 0 max = 10
Technical Interview Score: min = 0 max = 10


## 8. Descriptive statistics — numerical attributes (YOE, both scores)

In [9]:
df[['YOE', 'Code Challenge Score', 'Technical Interview Score']].describe()

,YOE,Code Challenge Score,Technical Interview Score
count,50000.000000,50000.000000,50000.000000
mean,15.286980,4.996400,5.003880
std,8.830652,3.166896,3.165082
min,0.000000,0.000000,0.000000
25%,8.000000,2.000000,2.000000
50%,15.000000,5.000000,5.000000
75%,23.000000,8.000000,8.000000
max,30.000000,10.000000,10.000000


## 9. Hiring rate under the business rule

In [10]:
is_hired_preview = (df['Code Challenge Score'] >= 7) & (df['Technical Interview Score'] >= 7)
print(f"Hired: {is_hired_preview.sum():,} ({is_hired_preview.mean()*100:.2f}%)")
print(f"Not hired: {(~is_hired_preview).sum():,} ({(1 - is_hired_preview.mean())*100:.2f}%)")

Hired: 6,698 (13.40%)
Not hired: 43,302 (86.60%)


## 10. Summary of Findings

- **Volume:** 50,000 records, 10 columns (one row per application).
- **Data types:** Application Date arrives as text (`YYYY-MM-DD`) and needs to be cast to datetime during the transformation phase. Scores and years of experience are already integers, and the rest are text.
- **Data quality:** Zero nulls across the entire dataset. No exact duplicate rows. There are 167 repeated emails (repeat applicants), but they are kept because the table is modeled at the *application* level, not unique candidates.
- **Date range:** From 2018-01-01 to 2022-07-04. Dates are clean, ideal for building `DimDate`.
- **Seniority:** 7 well-defined levels (`Trainee`, `Intern`, `Junior`, `Mid-Level`, `Senior`, `Lead`, `Architect`) with no text noise.
- **Technology & Country:** 24 unique technologies and 244 countries. Perfect for building dimensions (`DimTechnology` and `DimCountry`) and pulling metrics for queries.
- **Years of Experience (YOE):** Ranges from 0 to 30 years. We'll group them into bands (Entry, Junior, Mid, Senior, Expert) inside `DimCandidateProfile`.
- **Scores:** Code and technical interview scores are within the expected 0–10 range, with no negative or out-of-bounds values.
- **Key Business Rule:** Candidates are considered hired (`HIRED = 1`) if they score $\ge 7$ on both tests (code and technical). This logic will be formally applied in the transformation script.


## 10. Resumen del análisis exploratorio (Profiling)

- **Volumen:** 50,000 registros, 10 columnas (una fila por aplicación).
- **Tipos de datos:** La fecha de aplicación viene como texto (`YYYY-MM-DD`) y hay que convertirla a datetime en la fase de transformación. Los scores y años de experiencia ya están como enteros, y el resto son texto.
- **Calidad de datos:** Cero nulos en todo el dataset. No hay filas duplicadas exactas. Hay 167 correos repetidos (aplicantes recurrentes), pero se conservan porque la tabla se modela a nivel de *aplicación*, no de candidato único.
- **Rango temporal:** Del 2018-01-01 al 2022-07-04. Las fechas están limpias, perfecto para construir la `DimDate`.
- **Seniority:** 7 niveles bien definidos (`Trainee`, `Intern`, `Junior`, `Mid-Level`, `Senior`, `Lead`, `Architect`) sin ruido en los textos.
- **Tecnología y País:** 24 tecnologías y 244 países únicos. Sirven directo para armar las dimensiones (`DimTechnology` y `DimCountry`) y sacar las métricas de las consultas.
- **Años de experiencia (YOE):** Van de 0 a 30 años. Los agruparemos por rangos (Entry, Junior, Mid, Senior, Expert) en la `DimCandidateProfile`.
- **Scores:** Las puntuaciones de código y entrevista técnica están en el rango esperado de 0 a 10, sin valores negativos ni fuera de rango.
- **Regla de negocio clave:** Los candidatos se consideran contratados (`HIRED = 1`) si sacan $\ge 7$ en ambas pruebas (código y técnica). Esta lógica se aplicará formalmente en el script de transformación.